In [0]:
# =============================================================================
# segmentation_full_reconciliation  -  READ ONLY. START-TO-FINISH proof that EVERY
# eligible case (raw AppealCase CaseType=1) lands in EXACTLY ONE bucket, with:
#   * ZERO droppage  (no case in raw but in no bucket = the 'Not sure?' gap)   -> RED FLAG
#   * ZERO duplication (no case in >1 bucket)                                  -> RED FLAG
#   * every bucket count == expected (Bella VM v6)                             -> RED FLAG on any diff
# Buckets: active (stg_segmentation_states) + FTA + UTA + FPA + TD (stg_*_filtered).
# NO TOLERANCE: the run PASSES only if orphans=0 AND duplication=0 AND every count matches
# AND raw_universe == distinct_union_of_buckets. Single print + downloadable Excel.
#
# WHY this exists: the earlier verification only checked overlaps + counts (presence-based),
# so 8 cases that fell to the SQL 'ELSE Not sure?' branch (in NO bucket) were invisible.
# This closes that hole by reconciling against the full raw universe.
# =============================================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
import io, base64, datetime, pandas as pd

# ---- config ----
RAW_TBL   = "hive_metastore.ariadm_active_appeals.raw_appealcase"   # universe: CaseType=1
CASETYPE  = 1
BUCKETS = {
 "active":"hive_metastore.ariadm_active_appeals.stg_segmentation_states",
 "FTA"   :"hive_metastore.ariadm_arm_fta.stg_appeals_filtered",
 "UTA"   :"hive_metastore.ariadm_arm_uta.stg_appeals_filtered",
 "FPA"   :"hive_metastore.ariadm_arm_fpa.stg_filepreservedcases_filtered",
 "TD"    :"hive_metastore.ariadm_arm_td.stg_td_filtered",
}
EXPECTED = { "active":6353, "FTA":116637, "UTA":8180, "FPA":429, "TD":1807980 }   # Bella VM v6 2026-08-20
ACTIVE_STATE_EXPECTED = {
 "appealSubmitted":43,"awaitingRespondentEvidence(a)":9,"awaitingRespondentEvidence(b)":42,
 "caseUnderReview":177,"decided(a)":1983,"decided(b)":21,"decision":331,"ended":419,
 "ftpaSubmitted(a)":301,"ftpaDecided":1231,"ftpaSubmitted(b)":124,"listing":436,
 "paymentPending":19,"prepareForHearing":1079,"reasonsForAppealSubmitted":109,"remitted":29,
}
KNOWN_DROP_MAX = 2   # ARIADM-376 (business to resolve). RED FLAG if orphans exceed this; listed regardless.

REPORT=[]; log=lambda *a: REPORT.append(" ".join(str(x) for x in a))
def logdf(df,n=60):
    try: REPORT.append(df._jdf.showString(n,0,False))
    except Exception as e: REPORT.append(f"  (render fail: {str(e)[:120]})")
def col_ci(cols,name): return next((c for c in cols if c.lower()==name.lower()),None)
_c=spark.read.option("multiline","true").json("dbfs:/configs/config.json")
env_name=_c.first()["env"].strip().lower(); lz_key=_c.first()["lz_key"].strip().lower()
KV=f"ingest{lz_key}-meta002-{env_name}"
cid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-ID"); csec=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-SECRET"); tid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-TENANT-ID")
for sa in [f"ingest{lz_key}curated{env_name}",f"ingest{lz_key}raw{env_name}"]:
    spark.conf.set(f"fs.azure.account.auth.type.{sa}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{sa}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{sa}.dfs.core.windows.net",cid)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{sa}.dfs.core.windows.net",csec)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{sa}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tid}/oauth2/token")

In [0]:
# ---- load universe + buckets, build membership ----
raw=spark.table(RAW_TBL); r_cc=col_ci(raw.columns,"CaseNo"); r_ct=col_ci(raw.columns,"CaseType")
r_dp=col_ci(raw.columns,"DeptId") or col_ci(raw.columns,"DepartmentId"); r_cp=col_ci(raw.columns,"CasePrefix")
U=(raw.filter(col(r_ct)==CASETYPE).select(trim(col(r_cc)).alias("CaseNo")).dropDuplicates(["CaseNo"]))
uni_total=U.count()

def bucket_cases(tbl):
    t=spark.table(tbl); cc=col_ci(t.columns,"CaseNo")
    return t.select(trim(col(cc)).alias("CaseNo")).dropDuplicates(["CaseNo"])
memb=None
for b,tbl in BUCKETS.items():
    d=bucket_cases(tbl).withColumn("bucket",lit(b))
    memb=d if memb is None else memb.unionByName(d)
memb=memb.cache()
union_total=memb.select("CaseNo").distinct().count()

# per-case bucket count/set
per_case=memb.groupBy("CaseNo").agg(countDistinct("bucket").alias("nb"), collect_set("bucket").alias("buckets")).cache()

log("="*84); log("SEGMENTATION FULL RECONCILIATION  (zero-droppage, start->finish)"); log("="*84)
log(f"START  raw universe (CaseType={CASETYPE}) : {uni_total}")
log(f"END    distinct union of all buckets      : {union_total}")

In [0]:
# ---- 1) per-bucket count vs expected ----
log("\n"+"-"*84); log("1) PER-BUCKET COUNT vs EXPECTED (Bella VM v6)"); log("-"*84)
bucket_rows=[]
for b in BUCKETS:
    n=memb.filter(col("bucket")==b).select("CaseNo").distinct().count()
    exp=EXPECTED.get(b); diff=(n-exp) if exp is not None else None
    st="MATCH" if diff==0 else "*** DIFF ***"
    log(f"  {b:8s} actual={n:>9}  expected={exp:>9}  diff={diff:>+6}  {st}")
    bucket_rows.append({"bucket":b,"actual":n,"expected":exp,"diff":diff,"status":("MATCH" if diff==0 else "DIFF")})

# ---- 2) DUPLICATION: any case in >1 bucket ----
dup=per_case.filter(col("nb")>1).cache()
ndup=dup.count()
log("\n"+"-"*84); log(f"2) DUPLICATION (case in >1 bucket) : {ndup}   <<< MUST BE 0"); log("-"*84)
dup_rows=[]
if ndup:
    combo=dup.withColumn("combo",array_join(array_sort(col("buckets")),"+")).groupBy("combo").count().orderBy(desc("count"))
    logdf(combo,40)
    for r in dup.withColumn("combo",array_join(array_sort(col("buckets")),"+")).select("CaseNo","combo").limit(200).collect():
        dup_rows.append({"CaseNo":r["CaseNo"],"in_buckets":r["combo"]})

# ---- 3) ORPHAN / DROPPAGE: universe case in NO bucket ----
orph=U.join(memb.select("CaseNo").distinct(),"CaseNo","left_anti").cache()
norph=orph.count()
log("\n"+"-"*84); log(f"3) ORPHAN / DROPPED (in raw universe, in NO bucket) : {norph}   <<< MUST BE 0"); log("-"*84)
orph_rows=[]
if norph:
    od=(orph.join(raw.select(trim(col(r_cc)).alias("CaseNo"),
                             col(r_ct).alias("CaseType"),
                             (col(r_dp) if r_dp else lit(None)).alias("DeptId"),
                             (col(r_cp) if r_cp else lit(None)).alias("CasePrefix")),"CaseNo","left"))
    logdf(od.limit(200),200)
    for r in od.limit(500).collect():
        orph_rows.append({"CaseNo":r["CaseNo"],"CaseType":r["CaseType"],"DeptId":r["DeptId"],"CasePrefix":r["CasePrefix"]})

# ---- 4) EXTRA: in a bucket but NOT in raw CaseType=1 universe ----
extra=memb.select("CaseNo").distinct().join(U,"CaseNo","left_anti").cache()
nextra=extra.count()
log("\n"+"-"*84); log(f"4) EXTRA (in a bucket but not in raw CaseType={CASETYPE}) : {nextra}   <<< should be 0"); log("-"*84)
extra_rows=[]
if nextra:
    ed=extra.join(memb,"CaseNo","left").groupBy("bucket").count().orderBy(desc("count"))
    logdf(ed,20)
    for r in extra.join(memb,"CaseNo","left").select("CaseNo","bucket").limit(200).collect():
        extra_rows.append({"CaseNo":r["CaseNo"],"in_bucket":r["bucket"]})

# ---- 5) active state breakdown vs expected ----
log("\n"+"-"*84); log("5) ACTIVE per-state vs expected"); log("-"*84)
act=spark.table(BUCKETS["active"]); a_ts=col_ci(act.columns,"TargetState")
state_rows=[]
if a_ts:
    ac=act.groupBy(col(a_ts).alias("state")).agg(countDistinct(trim(col(col_ci(act.columns,"CaseNo")))).alias("actual"))
    amap={r["state"]:r["actual"] for r in ac.collect()}
    for s,exp in ACTIVE_STATE_EXPECTED.items():
        a=amap.get(s,0); d=a-exp
        log(f"  {s:32s} actual={a:>6} expected={exp:>6} diff={d:>+5} {'MATCH' if d==0 else '*** DIFF ***'}")
        state_rows.append({"state":s,"actual":a,"expected":exp,"diff":d,"status":("MATCH" if d==0 else "DIFF")})

In [0]:
# ---- VERDICT + Excel + single print ----
counts_ok = all(r["status"]=="MATCH" for r in bucket_rows)
states_ok = all(r["status"]=="MATCH" for r in state_rows) if state_rows else True
recon_ok  = (uni_total==union_total)
PASS = (ndup==0 and norph==0 and nextra==0 and counts_ok and states_ok and recon_ok)
log("\n"+"="*84)
log("RECON EQUATION  raw_universe == distinct_union :  "
    f"{uni_total} == {union_total}  -> {'OK' if recon_ok else 'MISMATCH ('+str(uni_total-union_total)+')'}")
log(f">>> VERDICT: {'PASS - every case in exactly one bucket, zero dropped, zero duplicated, all counts match' if PASS else 'FAIL'}")
if not PASS:
    reasons=[]
    if ndup: reasons.append(f"{ndup} duplicated")
    if norph: reasons.append(f"{norph} DROPPED/orphan")
    if nextra: reasons.append(f"{nextra} extra (bucket not in universe)")
    if not counts_ok: reasons.append("bucket count(s) != expected")
    if not states_ok: reasons.append("active state(s) != expected")
    if not recon_ok: reasons.append("universe != union")
    log("    reasons: "+"; ".join(reasons))
    log("    NOTE: ANY dropped case = a 'Not sure?' gap. No allowance. Chase every one.")
log("="*84)

import pandas as pd
def _pd(rows): return pd.DataFrame(rows) if rows else pd.DataFrame({"_":["(none)"]})
summary=pd.DataFrame([{
 "raw_universe_CaseType1":uni_total, "distinct_union_of_buckets":union_total,
 "duplicated":ndup, "dropped_orphan":norph, "extra_not_in_universe":nextra,
 "bucket_counts_all_match":counts_ok, "active_states_all_match":states_ok,
 "recon_universe==union":recon_ok, "VERDICT":("PASS" if PASS else "FAIL")}])
sheets={"summary":summary,"per_bucket":_pd(bucket_rows),"duplicated":_pd(dup_rows),
        "dropped_orphans":_pd(orph_rows),"extra":_pd(extra_rows),"active_states":_pd(state_rows)}
try:
    try: import openpyxl
    except Exception:
        import subprocess,sys; subprocess.run([sys.executable,"-m","pip","install","-q","openpyxl"])
    buf=io.BytesIO()
    with pd.ExcelWriter(buf,engine="openpyxl") as xw:
        for nm,pdf in sheets.items(): pdf.to_excel(xw,sheet_name=nm[:31],index=False)
    b64=base64.b64encode(buf.getvalue()).decode(); stamp=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    displayHTML(f'<a download="segmentation_full_reconciliation_{stamp}.xlsx" style="display:inline-block;background:#0b5cad;color:#fff;text-decoration:none;padding:8px 14px;border-radius:6px;font-family:sans-serif;font-size:13px" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">&#11015; Download full reconciliation ({len(sheets)} sheets)</a>')
except Exception as e: log(f"(download note: {str(e)[:120]})")
full="\n".join(REPORT)
try:
    user=spark.sql("SELECT current_user()").first()[0]; ts=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/segmentation_full_reconciliation/{ts}"; dbutils.fs.mkdirs(f"file:{folder}")
    p=f"{folder}/segmentation_full_reconciliation.txt"; open(p,"w").write(full); full+=f"\n\n>>> saved to: {p}"
except Exception as e: full+=f"\n(save note: {str(e)[:80]})"
print(full)